# 🛍️ Mall Customer Segmentation using K-Means (Unsupervised Learning)

Customer segmentation is the commercial practice of dividing a customer base into discrete groups that share similar demographic, behavioral, and spending characteristics.

This project covers:
- **Exploratory Data Analysis** of shopper demographics (Gender, Age, Income, Spending Score)
- **Optimal Cluster Identification**: The Elbow Method (WCSS) & Silhouette Analysis
- **K-Means 2D Segmentation**: Modeling Annual Income vs. Spending Score
- **Customer Persona Profiling**: Business interpretation of the 5 discovered segments
- **Downstream Classification**: Benchmarking supervised models (KNN, Naive Bayes, Decision Tree, Random Forest) to classify newly acquired customers into segments


### 1. Library Imports
Importing `pandas`, `numpy`, `seaborn`, `matplotlib.pyplot`, `scikit-learn` clustering algorithms, preprocessing tools, and classification estimators.


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, accuracy_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

import warnings
warnings.filterwarnings('ignore')


### 2. Loading the Dataset
Load `Mall_Customers.csv` into a pandas DataFrame and preview the first records.


In [ ]:
data = pd.read_csv("Mall_Customers.csv")
df = pd.DataFrame(data)
df.head()


### 3. Structural Overview: Dimensions & Data Types
Inspect dataset shape, total element count, and column datatypes.


In [ ]:
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Total Elements: {df.size}")
print("\n--- Column Information ---")
print(df.dtypes)
print("\n--- Summary Statistics ---")
df.describe()


### 4. Data Quality Check: Null Values & Duplicates
Verify that there are no missing values or duplicate rows in the customer dataset.


In [ ]:
print("Missing Values per Column:")
print(df.isna().sum())
print(f"\nDuplicate Records: {df.duplicated().sum()}")


### 5. Demographic Analysis: Gender Distribution
Visualize the proportion of male vs. female shoppers using a pie chart.


In [ ]:
gender_counts = df['Gender'].value_counts()

plt.figure(figsize=(6, 6))
plt.pie(
    gender_counts.values,
    labels=gender_counts.index,
    autopct='%1.1f%%',
    startangle=140,
    colors=['coral', 'steelblue']
)
plt.title('Customer Gender Distribution')
plt.show()

print(gender_counts)


### 6. Univariate Distributions: Age, Income, and Spending Score
Examine the empirical distributions and spreads of `Age`, `Annual Income (k$)`, and `Spending Score (1-100)`.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

sns.histplot(df['Age'], kde=True, ax=axes[0], color='teal')
axes[0].set_title('Age Distribution')

sns.histplot(df['Annual Income (k$)'], kde=True, ax=axes[1], color='darkorange')
axes[1].set_title('Annual Income Distribution (k$)')

sns.histplot(df['Spending Score (1-100)'], kde=True, ax=axes[2], color='purple')
axes[2].set_title('Spending Score Distribution (1-100)')

plt.tight_layout()
plt.show()


### 7. Feature Selection for Customer Segmentation
Isolate the two core behavioral features: `Annual Income (k$)` and `Spending Score (1-100)`. These represent the economic capacity and shopping behavior of each individual.


In [ ]:
X = df[['Annual Income (k$)', 'Spending Score (1-100)']]
print(f"Selected Feature Space Shape: {X.shape}")
X.head()


### 8. Finding Optimal $k$: The Elbow Method
Calculate the Within-Cluster Sum of Squares (WCSS / Inertia) across candidate cluster counts $k \in [1, 10]$ to spot the inflection point (elbow).


In [ ]:
wcss = []
k_range = range(1, 11)

for k in k_range:
    km = KMeans(n_clusters=k, init='k-means++', random_state=42)
    km.fit(X)
    wcss.append(km.inertia_)

plt.figure(figsize=(8, 4.5))
plt.plot(k_range, wcss, color='black', marker='o', linewidth=2)
plt.title('The Elbow Method for Mall Customer Segmentation')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('WCSS (Inertia)')
plt.xticks(k_range)
plt.grid(True, linestyle='--', alpha=0.5)
plt.axvline(x=5, color='crimson', linestyle='--', label='Optimal Elbow Point (k = 5)')
plt.legend()
plt.show()


### 9. Validating Cluster Cohesion: Silhouette Analysis
Evaluate Silhouette Scores across $k \in [2, 9]$ to measure how well-separated and cohesive the clusters are.


In [ ]:
sil_scores = []
k_sil_range = range(2, 10)

for k in k_sil_range:
    km = KMeans(n_clusters=k, init='k-means++', random_state=42)
    preds = km.fit_predict(X)
    score = silhouette_score(X, preds)
    sil_scores.append(score)
    print(f"k = {k}: Silhouette Score = {score:.4f}")

plt.figure(figsize=(8, 4))
plt.plot(k_sil_range, sil_scores, marker='s', color='navy', linewidth=2)
plt.title('Silhouette Score Analysis Across Cluster Counts')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Silhouette Coefficient')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()


### 10. Training the Final K-Means Model ($k = 5$)
Fit the K-Means algorithm with the optimal $k=5$ clusters using `k-means++` initialization and extract cluster labels and centroids.


In [ ]:
optimal_k = 5
kmeans = KMeans(n_clusters=optimal_k, init='k-means++', random_state=42)
df['Cluster'] = kmeans.fit_predict(X)
centroids = kmeans.cluster_centers_

print(f"Optimal Silhouette Score: {silhouette_score(X, df['Cluster']):.4f}")
print("\nCluster Distribution:")
print(df['Cluster'].value_counts().sort_index())


### 11. Visualizing Customer Segments in 2D Feature Space
Plot `Annual Income (k$)` vs. `Spending Score (1-100)` colored by assigned cluster, highlighting the five cluster centroids.


In [ ]:
plt.figure(figsize=(10, 6.5))

colors = ['royalblue', 'forestgreen', 'darkorange', 'crimson', 'purple']
cluster_names = [
    'Cluster 0: Careful / Savers',
    'Cluster 1: Standard / Average',
    'Cluster 2: Target / High Spenders',
    'Cluster 3: Careless / Impulsive',
    'Cluster 4: Sensible / Low Income'
]

for i in range(optimal_k):
    cluster_data = df[df['Cluster'] == i]
    plt.scatter(
        cluster_data['Annual Income (k$)'],
        cluster_data['Spending Score (1-100)'],
        s=60, color=colors[i], label=f'Cluster {i}'
    )

# Plot Centroids
plt.scatter(
    centroids[:, 0], centroids[:, 1],
    s=250, c='black', marker='X', edgecolor='white', linewidth=2, label='Centroids'
)

plt.title('Mall Customer Segments (K-Means Clustering, k = 5)', fontsize=14)
plt.xlabel('Annual Income (k$)', fontsize=12)
plt.ylabel('Spending Score (1-100)', fontsize=12)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.grid(True, linestyle='--', alpha=0.4)
plt.show()


### 12. Customer Persona Profiling & Business Insights
Aggregate average metrics per cluster to formulate strategic business personas:
1. **Target Customers** (High Income, High Spending): Premium shoppers; prime targets for luxury campaigns, VIP loyalty programs, and high-margin product launches.
2. **Careful Shoppers / Savers** (High Income, Low Spending): High purchasing power but conservative spenders; engage via value-driven propositions, quality guarantees, and investment-grade promotions.
3. **Standard / Moderate** (Mid Income, Mid Spending): The steady customer core; responsive to seasonal discounts and mainstream marketing.
4. **Careless / Impulsive** (Low Income, High Spending): Enthusiastic spenders despite modest earnings; responsive to trend-driven retail, impulse promotions, and easy financing/installment schemes.
5. **Sensible Shoppers** (Low Income, Low Spending): Budget-conscious customers; prioritize budget deals, essential goods, and value discounts.


In [ ]:
persona_summary = df.groupby('Cluster').agg({
    'Age': 'mean',
    'Annual Income (k$)': 'mean',
    'Spending Score (1-100)': 'mean',
    'CustomerID': 'count'
}).rename(columns={'CustomerID': 'Customer Count'}).reset_index()

persona_summary


### 13. Supervised Classification: Setup & Preprocessing
To assign newly acquired customers into these behavioral segments automatically, we train supervised classification models using `MinMaxScaler` normalization on an 70/30 train/test split.


In [ ]:
input_features = df[['Annual Income (k$)', 'Spending Score (1-100)']].values
output_cluster = df['Cluster'].values

X_train, X_test, y_train, y_test = train_test_split(
    input_features, output_cluster, test_size=0.3, random_state=42
)

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}")


### 14. Multi-Model Benchmark: KNN, Naive Bayes, Decision Tree, and Random Forest
Fit four supervised algorithms to learn the cluster boundaries and compare test set accuracy scores.


In [ ]:
# 1. K-Nearest Neighbors
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
knn_preds = knn.predict(X_test_scaled)
score_knn = accuracy_score(y_test, knn_preds)

# 2. Gaussian Naive Bayes
nb = GaussianNB()
nb.fit(X_train_scaled, y_train)
nb_preds = nb.predict(X_test_scaled)
score_nb = accuracy_score(y_test, nb_preds)

# 3. Decision Tree Classifier
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train_scaled, y_train)
dt_preds = dt.predict(X_test_scaled)
score_tree = accuracy_score(y_test, dt_preds)

# 4. Random Forest Classifier
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_scaled, y_train)
rf_preds = rf.predict(X_test_scaled)
score_rf = accuracy_score(y_test, rf_preds)

comparison_df = pd.DataFrame({
    'model_name': ['K-Nearest Neighbors', 'Naive Bayes', 'Decision Tree', 'Random Forest'],
    'accuracy_score': [score_knn, score_nb, score_tree, score_rf]
}).sort_values(by='accuracy_score', ascending=False).reset_index(drop=True)

comparison_df


### 15. Visual Performance Comparison
Visualize accuracy scores across all 4 supervised classifiers using a bar chart with explicit value annotations.


In [ ]:
plt.figure(figsize=(8, 4.5))
bars = plt.bar(comparison_df['model_name'], comparison_df['accuracy_score'], color='royalblue', width=0.55)
plt.title('Classifier Accuracy Comparison for Predicting Customer Clusters', fontsize=12)
plt.ylabel('Accuracy Score', fontsize=11)
plt.ylim(0.8, 1.05)

for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2, height + 0.008,
        f"{height:.3f}", ha='center', va='bottom', fontweight='bold'
    )

plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.show()


### 16. Key Takeaways & Strategic Summary
1. **Clear Geometric Separation**: Annual Income and Spending Score partition into 5 natural clusters with very high boundaries, evidenced by near-perfect downstream classifier accuracy.
2. **Actionable Commercial Value**: Retail managers can tailor inventory, marketing budgets, and loyalty reward tiers to specific shopper personas rather than treating the customer base as a monolith.
3. **Automated Onboarding**: The trained supervised classifier allows the business to immediately tag new loyalty club members into their respective segment upon capturing their basic financial and spending indicators.
